# 🌾 Smart Agri City — EDA & Model Training Notebook
**Service:** Python ML Service | **Port:** 5000

Notebook ini mencakup:
- EDA: distribusi data, korelasi, outlier, class balance
- Preprocessing: StandardScaler, LabelEncoder, train-test split 80/20
- Training 3 model + evaluasi metrics (R², Accuracy)
- Cross-validation 5-fold
- Feature importance plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid', palette='husl')
print('✅ Libraries loaded successfully!')

---
## 1. Load Datasets

In [ ]:
df_yield = pd.read_csv('data/crop_yield.csv')
df_pest  = pd.read_csv('data/pest_disease.csv')
df_irrig = pd.read_csv('data/irrigation_demand.csv')

print('crop_yield     :', df_yield.shape)
print('pest_disease   :', df_pest.shape)
print('irrigation_demand:', df_irrig.shape)

display(df_yield.head(3))
display(df_pest.head(3))
display(df_irrig.head(3))

---
## 2. EDA — Distribusi Fitur

In [ ]:
# ── Crop Yield feature distributions ──
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
numeric_cols = df_yield.select_dtypes(include=np.number).columns
for ax, col in zip(axes.flatten(), numeric_cols):
    ax.hist(df_yield[col], bins=30, edgecolor='white', color='steelblue')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Crop Yield Dataset — Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Irrigation feature distributions ──
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
irrig_num = df_irrig.select_dtypes(include=np.number).columns
for ax, col in zip(axes.flatten(), irrig_num):
    ax.hist(df_irrig[col], bins=30, edgecolor='white', color='teal')
    ax.set_title(col, fontsize=10)
plt.suptitle('Irrigation Dataset — Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. EDA — Boxplot Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df_yield[['soil_moisture','nitrogen','yield_ton_per_ha']].boxplot(ax=axes[0])
axes[0].set_title('Yield — Key Features Boxplot')

df_pest[['air_humidity','leaf_temp','chlorophyll']].boxplot(ax=axes[1])
axes[1].set_title('Pest — Key Features Boxplot')

df_irrig[['soil_moisture','water_needed_liters','irrigation_urgency']].boxplot(ax=axes[2])
axes[2].set_title('Irrigation — Key Features Boxplot')

plt.suptitle('Outlier Detection — Boxplots per Dataset', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. EDA — Correlation Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(df_yield.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', ax=axes[0], linewidths=.5)
axes[0].set_title('Crop Yield — Correlation Heatmap')

sns.heatmap(df_irrig.drop(columns=['growth_phase']).corr(numeric_only=True),
            annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], linewidths=.5)
axes[1].set_title('Irrigation — Correlation Heatmap')

plt.tight_layout()
plt.show()

---
## 5. EDA — Class Balance Check (Pest Dataset)

In [ ]:
counts = df_pest['pest_category'].value_counts()
pct    = df_pest['pest_category'].value_counts(normalize=True) * 100

print('=== Pest Category Distribution ===')
for cat in counts.index:
    print(f'  {cat:<20} {counts[cat]:>5} rows  ({pct[cat]:.1f}%)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
counts.plot(kind='bar', ax=ax1, color='coral', edgecolor='white')
ax1.set_title('Pest Category — Count')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=30)

ax2.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
ax2.set_title('Pest Category — Proportion')
plt.tight_layout()
plt.show()

---
## 6. Preprocessing

In [ ]:
# ── MODEL 1: Crop Yield — preprocessing ──
YIELD_FEATS = ['avg_temp','rainfall','soil_moisture','ph',
               'nitrogen','phosphorus','potassium','area_ha','week_of_planting']

scaler_y = StandardScaler()
X_y = scaler_y.fit_transform(df_yield[YIELD_FEATS])
y_y = df_yield['yield_ton_per_ha'].values
X_y_tr, X_y_te, y_y_tr, y_y_te = train_test_split(X_y, y_y, test_size=0.2, random_state=42)
print(f'Yield  — Train: {X_y_tr.shape[0]} | Test: {X_y_te.shape[0]}')

# ── MODEL 2: Pest Classifier — preprocessing ──
PEST_FEATS = ['air_humidity','leaf_temp','soil_ph','chlorophyll','light_lux','zone_enc']
le_zone = LabelEncoder()
le_pest = LabelEncoder()
df_pest['zone_enc'] = le_zone.fit_transform(df_pest['zone'])
y_p = le_pest.fit_transform(df_pest['pest_category'])

scaler_p = StandardScaler()
X_p = scaler_p.fit_transform(df_pest[PEST_FEATS])
X_p_tr, X_p_te, y_p_tr, y_p_te = train_test_split(X_p, y_p, test_size=0.2, random_state=42)
print(f'Pest   — Train: {X_p_tr.shape[0]} | Test: {X_p_te.shape[0]}')

# ── MODEL 3: Irrigation Optimizer — preprocessing ──
IRR_FEATS = ['soil_moisture','air_temp','rain_forecast','growth_phase_enc','evapotranspiration']
le_phase = LabelEncoder()
df_irrig['growth_phase_enc'] = le_phase.fit_transform(df_irrig['growth_phase'])
y_i = df_irrig['water_needed_liters'].values

scaler_i = StandardScaler()
X_i = scaler_i.fit_transform(df_irrig[IRR_FEATS])
X_i_tr, X_i_te, y_i_tr, y_i_te = train_test_split(X_i, y_i, test_size=0.2, random_state=42)
print(f'Irrig  — Train: {X_i_tr.shape[0]} | Test: {X_i_te.shape[0]}')

---
## 7. Training & Evaluasi Model

In [ ]:
# ── MODEL 1: Crop Yield Predictor ──
mdl_y = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42)
mdl_y.fit(X_y_tr, y_y_tr)

r2_test_y = r2_score(y_y_te, mdl_y.predict(X_y_te))
cv_y = cross_val_score(mdl_y, X_y, y_y, cv=5, scoring='r2')
print(f'[Yield]      R² Test : {r2_test_y:.4f} | 5-Fold CV: {cv_y.mean():.4f} ± {cv_y.std():.4f}')
assert cv_y.mean() >= 0.70, '❌ Yield R² below target!'
print('  ✅ Target ≥ 0.70 PASSED')

In [ ]:
# ── MODEL 2: Pest & Disease Classifier ──
mdl_p = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=42)
mdl_p.fit(X_p_tr, y_p_tr)

acc_test_p = accuracy_score(y_p_te, mdl_p.predict(X_p_te))
cv_p = cross_val_score(mdl_p, X_p, y_p, cv=5, scoring='accuracy')
print(f'[Pest]       Acc Test: {acc_test_p:.4f} | 5-Fold CV: {cv_p.mean():.4f} ± {cv_p.std():.4f}')
print(classification_report(y_p_te, mdl_p.predict(X_p_te),
                             target_names=le_pest.classes_))
assert cv_p.mean() >= 0.70, '❌ Pest Accuracy below target!'
print('  ✅ Target ≥ 0.70 PASSED')

In [ ]:
# ── MODEL 3: Irrigation Optimizer ──
mdl_i = GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, random_state=42)
mdl_i.fit(X_i_tr, y_i_tr)

r2_test_i = r2_score(y_i_te, mdl_i.predict(X_i_te))
cv_i = cross_val_score(mdl_i, X_i, y_i, cv=5, scoring='r2')
print(f'[Irrigation] R² Test : {r2_test_i:.4f} | 5-Fold CV: {cv_i.mean():.4f} ± {cv_i.std():.4f}')
assert cv_i.mean() >= 0.70, '❌ Irrigation R² below target!'
print('  ✅ Target ≥ 0.70 PASSED')

---
## 8. Feature Importance Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Yield
fi_y = pd.Series(mdl_y.feature_importances_, index=YIELD_FEATS).sort_values(ascending=True)
fi_y.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Crop Yield — Feature Importance')
axes[0].set_xlabel('Importance')

# Pest
fi_p = pd.Series(mdl_p.feature_importances_, index=PEST_FEATS).sort_values(ascending=True)
fi_p.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Pest Classifier — Feature Importance')
axes[1].set_xlabel('Importance')

# Irrigation
fi_i = pd.Series(mdl_i.feature_importances_, index=IRR_FEATS).sort_values(ascending=True)
fi_i.plot(kind='barh', ax=axes[2], color='teal')
axes[2].set_title('Irrigation Optimizer — Feature Importance')
axes[2].set_xlabel('Importance')

plt.suptitle('Feature Importance — All 3 Models', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Confusion Matrix — Pest Classifier

In [ ]:
cm = confusion_matrix(y_p_te, mdl_p.predict(X_p_te))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_pest.classes_,
            yticklabels=le_pest.classes_)
plt.title('Pest Classifier — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## 10. Save Models

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Save using same structure as train_models.py
le_zone2  = LabelEncoder().fit(df_pest['zone'])
le_pest2  = LabelEncoder().fit(df_pest['pest_category'])
le_phase2 = LabelEncoder().fit(df_irrig['growth_phase'])

# Re-encode for save-compatible model
df_pest2 = df_pest.copy()
df_pest2['zone'] = le_zone2.transform(df_pest2['zone'])
df_pest2['pest_category'] = le_pest2.transform(df_pest2['pest_category'])
X_pest_final = df_pest2.drop(columns=['pest_category','zone_enc'])
y_pest_final = df_pest2['pest_category']

df_irrig2 = df_irrig.copy()
df_irrig2['growth_phase'] = le_phase2.transform(df_irrig2['growth_phase'])
X_irrig_final = df_irrig2.drop(columns=['water_needed_liters','irrigation_urgency','growth_phase_enc'])
y_irrig_final = df_irrig2[['water_needed_liters','irrigation_urgency']]

X_yield_final = df_yield.drop(columns=['yield_ton_per_ha'])
y_yield_final = df_yield['yield_ton_per_ha']

mdl_y_final = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42)
mdl_y_final.fit(X_yield_final, y_yield_final)

mdl_p_final = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=42)
mdl_p_final.fit(X_pest_final, y_pest_final)

mdl_i_final = RandomForestRegressor(n_estimators=100, random_state=42)
mdl_i_final.fit(X_irrig_final, y_irrig_final)

payload = {
    'model_yield': mdl_y_final,
    'model_pest' : mdl_p_final,
    'model_irrig': mdl_i_final,
    'encoders': {
        'zone'        : le_zone2,
        'pest_category': le_pest2,
        'growth_phase' : le_phase2,
    },
    'features': {
        'yield'     : list(X_yield_final.columns),
        'pest'      : list(X_pest_final.columns),
        'irrigation': list(X_irrig_final.columns),
    }
}
joblib.dump(payload, 'models/agri_models.pkl')
print('✅ Models saved → models/agri_models.pkl')
print('Keys:', list(payload.keys()))